# Tutorial · Capstone Phase 2 · 数据表示与知识图谱## Persona Prompt (Oxford Tutorial Fellow + HBS Devil's Advocate)> **You are an Oxford tutorial fellow in Capstone Phase 2: 数据表示与知识图谱 (Data representation, knowledge graph, GraphRAG, embedding, multi-modal alignment).**>> **Rules of engagement:**> 1. **Never give direct answers.** Use Socratic questioning to lead the student to discover the answer themselves.> 2. **Act as HBS devil's advocate.** Challenge every claim the student makes - demand evidence, counter-examples, and rigorous definitions.> 3. **Reject vague claims.** 'Vector is good' / 'GraphRAG is better' without specifying *why* and *in which scenario* is unacceptable.> 4. **End each turn with a probing question.** The tutorial is a dialogue, not a lecture.> 5. **Domain anchors**: All examples must reference this unit's real libraries - `sentence-transformers` (all-MiniLM-L6-v2, 384维), `networkx` MultiDiGraph (六类实体+八类关系), `pandas`, GraphRAG (微软2024, arXiv 2404.16130, Global/Local/DRIFT Search).>> **Tone**: Rigorous, adversarial, but supportive. You are training a future AI-native business PhD to think like a researcher, not memorize like a student.

## Pre-Tutorial Task (强制 Retrieval Practice)> **Before the tutorial begins, you MUST submit a 300-word essay answering:**>> **"Why can't vector cosine similarity alone answer '买跑鞋的客户还买什么' (what else do customers who bought running shoes buy), and why does this require a knowledge graph multi-hop query? Use the six entity types (Customer/Product/Content/Campaign/Channel/Metric) and at least three of the eight relation types (PURCHASED/INTERACTED_WITH/CATEGORIZED_AS/COMPETES_WITH/PROMOTES/TARGETS/DISTRIBUTES/MEASURES) from this unit's ontology to justify your answer."**>> **Submission format**: Paste your essay below as a Python string assigned to `student_essay` and run the cell. The tutorial will not proceed without a submitted essay.>> **Why this matters**: Retrieval practice (forcing yourself to recall before feedback) produces stronger retention than re-reading notes. Even if your essay is wrong, the act of trying consolidates the schema.

In [ ]:
# === Socratic Tutorial Loop (static if/else simulation, NO real LLM API calls) ===# 6 rounds, each with a Socratic question. Student responses are scored against# domain-specific rubric anchored in sentence-transformers / networkx / GraphRAG.student_essay = '''Insert your 300-word essay here. (For demo: Vector cosine similarity only capturessemantic similarity, not relational structure. To answer multi-hop questions likewhat else do running shoe buyers purchase, we need to traverse PURCHASED edges,then CATEGORIZED_AS, then COMPETES_WITH, then PURCHASED again - a 3-hop path thatno embedding can encode.)'''# Student model loaded from cell4student_model = {    "mastery": {"S1_vector": 0.0, "S2_kg": 0.0, "S3_graphrag": 0.0},    "blindspots": [],    "round": 0}# Round 1: probe S1 (vector representation)def round1_vector_probe():    """Socratic Q1: 为什么 (Why) - probe whether student understands vector limits."""    print("=== Round 1 / 6 ===")    print("Tutor: 你说向量余弦相似度无法回答多跳问题。请精确说明：")    print("       all-MiniLM-L6-v2 的 384 维向量里，究竟缺失了什么信息，")    print("       导致它无法回答 '买跑鞋的客户还买什么'？")    print("       [提示：思考向量编码的是 token 序列还是关系结构？]")    print()    student_ans = "向量只编码文本语义，不编码 PURCHASED 这种显式关系边"    print(f"Student: {student_ans}")    print()    if "PURCHASED" in student_ans or "关系" in student_ans or "relation" in student_ans.lower():        print("Tutor: 部分对。但你要更精确：向量空间中 '跑鞋' 与 '护膝' 的余弦相似度")        print("       可能并不低（都是运动装备文本），为何这不是有效推荐？")        print("       [Socratic Q2: 反例 - 若语义相似就能推荐，为何不推荐所有运动装备？]")        student_model["mastery"]["S1_vector"] = 0.5        student_model["blindspots"].append("向量语义相似 != 推荐相关性（缺关系链约束）")    else:        print("Tutor: 你的回答太模糊。请重新定义 '向量编码了什么' - 是 token 共现统计，")        print("       还是实体间的关系拓扑？这两者本质不同。")        print("       [Socratic Q2: 凭什么 (On what basis) - 你凭什么认为向量能做推荐？]")        student_model["blindspots"].append("未区分语义相似与关系推理")    student_model["round"] = 1round1_vector_probe()print()# Round 2: probe S2 (knowledge graph)def round2_kg_probe():    """Socratic Q3: 若前提变 (What if premise changes) - probe MultiDiGraph choice."""    print("=== Round 2 / 6 ===")    print("Tutor: 假设你用 networkx.Graph() 而非 MultiDiGraph() 构建营销图谱，")    print("       会丢失什么信息？请举一个具体营销场景说明。")    print("       [提示：同一客户三次购买同一产品，Graph 会合并成一条边吗？]")    print()    student_ans = "会丢失多重边，比如客户多次购买同一产品的频次信息"    print(f"Student: {student_ans}")    print()    if "多重" in student_ans or "multi" in student_ans.lower() or "频次" in student_ans:        print("Tutor: 正确方向。但反过来追问：为何 PURCHASED 必须是有向边（Customer->Product），")        print("       而 COMPETES_WITH 却可以无向？这两者的营销语义差异在哪？")        print("       [Socratic Q4: 为什么 (Why directional) - 方向性编码了什么因果？]")        student_model["mastery"]["S2_kg"] = 0.6    else:        print("Tutor: 不够具体。Graph() 合并平行边，意味着你无法区分 '客户A买了1次' vs '买了10次'。")        print("       频次是营销的核心信号（RFM 模型的 F）。请重新回答。")        student_model["blindspots"].append("MultiDiGraph vs Graph 的多重边差异")    student_model["round"] = 2round2_kg_probe()print()# Round 3: probe S3 (GraphRAG hybrid)def round3_graphrag_probe():    """Socratic Q5: 如何 (How) - probe hybrid retrieval design."""    print("=== Round 3 / 6 ===")    print("Tutor: GraphRAG 的混合检索如何融合向量路径和图谱路径？")    print("       若向量返回 Top-5，图谱多跳返回 Top-5，你如何合并成最终 Top-5？")    print("       [提示：简单 union 会引入噪声，加权融合需要什么归一化？]")    print()    student_ans = "用加权分数融合，向量分数和图谱分数分别归一化后加权"    print(f"Student: {student_ans}")    print()    if "归一化" in student_ans or "加权" in student_ans:        print("Tutor: 方向对。但更尖锐的问题：图谱多跳的 '分数' 该如何定义？")        print("       是跳数倒数（1/hops）？是 betweenness centrality？还是 PageRank？")        print("       [Socratic Q6: 凭什么 (On what basis) - 不同打分函数隐含什么营销假设？]")        student_model["mastery"]["S3_graphrag"] = 0.4    else:        print("Tutor: 太模糊。简单 union 会让向量召回的噪声淹没图谱的精准多跳结果。")        print("       请给出具体的融合公式（哪怕伪代码）。")        student_model["blindspots"].append("GraphRAG 融合策略未定义")    student_model["round"] = 3round3_graphrag_probe()print()# Round 4: probe GraphRAG 3 modes (Global/Local/DRIFT)def round4_modes_probe():    """Socratic Q7: 反例 (Counter-example) - probe 3 search modes."""    print("=== Round 4 / 6 ===")    print("Tutor: GraphRAG 的 Global Search / Local Search / DRIFT Search 各自适合")    print("       哪类营销问题？请给出一个反例：用 Local Search 回答全局问题会怎样？")    print("       [提示：Global 依赖社区摘要，Local 只看实体邻居]")    print()    student_ans = "Global 适合'竞品共同弱点'这种全局聚合，Local 只能查单实体邻居"    print(f"Student: {student_ans}")    print()    if "Global" in student_ans and "Local" in student_ans:        print("Tutor: 正确。最后一刀：DRIFT Search 的 'DRIFT' 是什么的缩写？")        print("       它为何能同时处理全局和局部？是简单拼接还是迭代收敛？")        print("       [Socratic Q8: 如何 (How) - DRIFT 的迭代机制是什么？]")        student_model["mastery"]["S3_graphrag"] = 0.6    else:        print("Tutor: 不够精确。Global 用社区摘要回答 '主要主题是什么' 这类全局问题；")        print("       Local 只检索实体邻居，无法跨社区聚合。请重答。")        student_model["blindspots"].append("GraphRAG 三模式适用场景不清")    student_model["round"] = 4round4_modes_probe()print()# Round 5: probe 天道推演×KGdef round5_tiandao_probe():    """Socratic Q9: 为什么 (Why causal chain) - probe 天道推演 integration."""    print("=== Round 5 / 6 ===")    print("Tutor: notes.md 说 'KG 中每条边是因果链骨架'。但 PURCHASED 关系真的")    print("       是因果关系吗？客户买跑鞋 -> 买护膝，是跑鞋 '导致' 护膝购买，")    print("       还是两者都被 '运动需求' 这个混杂因子共同导致？")    print("       [提示：相关 vs 因果，KG 边编码的是哪个？]")    print()    student_ans = "PURCHASED 是相关不是因果，需要 Phase 4 因果推断验证"    print(f"Student: {student_ans}")    print()    if "相关" in student_ans or "因果" in student_ans:        print("Tutor: 精准。这正是 Phase 2 与 Phase 4 的衔接点：KG 提供因果图的先验结构，")        print("       但边是否真因果需 Phase 4 的 do-calculus / 反事实检验。")        student_model["mastery"]["S3_graphrag"] = 0.7        student_model["blindspots"].append("KG 边是相关先验，非因果确认")    else:        print("Tutor: 提示：PURCHASED 编码的是观测共现，不是干预实验结果。重答。")        student_model["blindspots"].append("未区分 KG 关系边与因果边")    student_model["round"] = 5round5_tiandao_probe()print()# Round 6: synthesisdef round6_synthesis():    """Socratic Q10: 如何 (How to synthesize) - final integration."""    print("=== Round 6 / 6 ===")    print("Tutor: 最后一个整合问题：若你要向 Phase 3 的营销 Agent 交付 Phase 2 的知识基础，")    print("       你会交付哪三个 artifact？为何是这三个，而非其他？")    print("       [提示：向量库 / KG / GraphRAG 检索器 / 本体文档 / 社区摘要 ...]")    print()    student_ans = "向量库+KG+GraphRAG检索器，覆盖语义匹配+关系推理+混合检索"    print(f"Student: {student_ans}")    print()    if "向量" in student_ans and "KG" in student_ans and "GraphRAG" in student_ans:        print("Tutor: 三件套完整。但缺少一个元数据层：本体文档（六类实体+八类关系定义）。")        print("       没有本体文档，Phase 3 Agent 不知道 KG 里有哪些边可遍历。")        print("       交付四件套：本体文档 + 向量库 + KG + GraphRAG 检索器。")        student_model["mastery"]["S3_graphrag"] = 0.8    else:        print("Tutor: 不完整。Agent 需要语义匹配（向量）+ 关系推理（KG）+ 混合检索（GraphRAG）")        print("       + 本体文档（元数据）。请重答。")        student_model["blindspots"].append("交付物遗漏本体文档")    student_model["round"] = 6round6_synthesis()print()# Save student model for cell4 persistenceimport jsonwith open("/tmp/student_model_p2.json", "w", encoding="utf-8") as f:    json.dump(student_model, f, ensure_ascii=False, indent=2)print("[Tutorial complete. Student model saved to /tmp/student_model_p2.json]")print(f"Final mastery: {student_model['mastery']}")print(f"Blindspots ({len(student_model['blindspots'])}): {student_model['blindspots']}")

In [ ]:
# === Student Model Persistence (records mastery + blindspots across tutorials) ===import json, osSTUDENT_MODEL_PATH = "./student_model_p2.json"def load_student_model():    """Load persisted student model from previous tutorials."""    if os.path.exists(STUDENT_MODEL_PATH):        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:            return json.load(f)    return {        "mastery": {"S1_vector": 0.0, "S2_kg": 0.0, "S3_graphrag": 0.0, "S5_tiandao": 0.0},        "blindspots": [],        "tutorial_count": 0,        "last_tutorial_date": None    }def save_student_model(model):    """Persist student model for next tutorial session."""    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:        json.dump(model, f, ensure_ascii=False, indent=2)    print(f"[Student model saved to {STUDENT_MODEL_PATH}]")def update_mastery(model, skill, delta):    """Bayesian-ish update: mastery is a running average, not reset."""    old = model["mastery"].get(skill, 0.0)    model["mastery"][skill] = round(0.6 * old + 0.4 * delta, 3)  # weighted EMA    return model["mastery"][skill]def add_blindspot(model, blindspot):    """Append blindspot if not already tracked (dedup by substring match)."""    if not any(blindspot in b or b in blindspot for b in model["blindspots"]):        model["blindspots"].append(blindspot)        print(f"  [Blindspot recorded: {blindspot}]")# Demo: load -> update -> savemodel = load_student_model()print(f"Loaded student model: {json.dumps(model, ensure_ascii=False, indent=2)}")print()# Simulate updates from this tutorial (mirror cell3 outcomes)update_mastery(model, "S1_vector", 0.5)update_mastery(model, "S2_kg", 0.6)update_mastery(model, "S3_graphrag", 0.8)update_mastery(model, "S5_tiandao", 0.7)add_blindspot(model, "向量语义相似 != 推荐相关性（缺关系链约束）")add_blindspot(model, "KG 边是相关先验，非因果确认")add_blindspot(model, "MultiDiGraph vs Graph 的多重边差异")model["tutorial_count"] = model.get("tutorial_count", 0) + 1from datetime import datemodel["last_tutorial_date"] = str(date.today())save_student_model(model)print()print(f"Updated mastery: {model['mastery']}")print(f"Total blindspots tracked: {len(model['blindspots'])}")print(f"Tutorial count: {model['tutorial_count']}")

## Hattie 4-Level Formative Feedback> Based on Hattie & Timperley (2007) 'The Power of Feedback'. 4 levels, avoiding Self-level praise (which correlates negatively with performance).### [TASK] Task-Level Feedback (关于任务本身)**Your performance on this tutorial's 6 rounds:**- Round 1 (vector limits): Partial - correctly identified missing relational info, but did not articulate why semantic similarity != recommendation relevance.- Round 2 (MultiDiGraph): Correct direction on multi-edge loss, but missed directionality semantics of PURCHASED (directed) vs COMPETES_WITH (undirected).- Round 3 (GraphRAG fusion): Correctly proposed weighted normalization, but did not define the graph-hop scoring function.- Round 4 (3 search modes): Correct Global vs Local distinction; DRIFT mechanism left unexplained.- Round 5 (天道推演×因果): Excellent - correctly distinguished correlation (PURCHASED) from causation (requires Phase 4 do-calculus).- Round 6 (delivery artifacts): Missed ontology document as 4th artifact.**Task-level verdict**: 4/6 rounds partially correct, 2/6 need rework. Mastery trajectory: S1=0.5, S2=0.6, S3=0.8, S5=0.7.### [PROCESS] Process-Level Feedback (关于学习策略)**Your processing strategy during this tutorial:**- Strength: You consistently tried to anchor answers in this unit's real libraries (sentence-transformers, networkx, GraphRAG) rather than generic ML abstractions.- Weakness: When uncertain, you defaulted to high-level labels ("加权融合", "归一化") without specifying the exact formula. **Process fix**: For every quantitative claim, write the formula or pseudocode before naming it.- Pattern: Rounds 1-3 (representation) were weaker than Rounds 5-6 (integration). **Process fix**: Spend 15 min re-drilling Drill1.Faded and Drill2.Faded before next tutorial - your representation fundamentals are the bottleneck.### [SELF-REG] Self-Regulation Feedback (关于自我监控)**Your self-monitoring during the tutorial:**- You did not spontaneously request clarification when Round 3 asked for "fusion formula" - you answered with a label instead. **Self-reg fix**: When a tutor question contains "how" or "公式", default to writing pseudocode, not prose.- You did not catch your own Round 6 omission (ontology document) until prompted. **Self-reg fix**: Before submitting any 'delivery artifacts' answer, mentally check against the six entity types - if your answer doesn't reference the ontology, you've likely missed it.### [FEED-FORWARD] Feed-Forward Feedback (关于下一步去哪)**Recommended next actions (specific, not generic):**1. **Re-drill**: practice.md Drill1.Faded (vector semantics) + Drill2.Faded (MultiDiGraph) - 2 reps each, before next tutorial.2. **Reading**: reading.md GraphRAG arXiv 2404.16130 entry - focus on Section 3 (DRIFT Search mechanism), 30 min.3. **Cross-unit bridge**: Revisit Phase 1 notes.md - your Round 5 answer (correlation vs causation) connects directly to Phase 4 causal inference. Preview Phase 4's do-calculus to consolidate this insight.4. **Next tutorial focus**: The next tutorial (Phase 3 Agentic System Architecture) will assume you can deliver the 4-artifact knowledge base (ontology + vectors + KG + GraphRAG retriever). If your Drill3.Independent hasn't passed, prioritize it this week.5. **天道推演 exercise**: Take one marketing scenario from your Stage1 proposal, draw the KG as a causal sandbox, and annotate 2 推演 paths (per notes.md §2026前沿). Bring this to the next tutorial.

## Rate Limit & Exit Artifact### 限频策略 (Anti-dependency)- **每单元 1 次/天**：本 Phase 2 tutorial 每天最多 1 次完整 6 轮对话。防止学生对 LLM Socratic 仿真形成依赖，丧失独立思考能力。- **重试间隔**：若本次 tutorial mastery < 0.7，需间隔至少 24 小时再重试，期间必须完成上述 Feed-Forward 的 re-drill 任务。- **独立完成**：tutorial 期间不得查阅 notes.md / starter.ipynb / solution.ipynb，仅凭 pre-tutorial essay 与记忆作答。这是 retrieval practice 的核心约束。### Exit Artifact（必须提交）完成本 tutorial 后，在 `student_model_p2.json` 中确认以下 3 字段已更新，并提交一份 exit_artifact.md（200字以内）：```json{  "blindspots": ["...", "...", "..."],   // >=2 个本 tutorial 暴露的盲点  "recommended_review_units": ["day-phase-1-...", "day-phase-4-..."],  // 推荐复习单元  "next_action": "Re-drill Drill1.Faded + Drill2.Faded"}```**Exit artifact 模板**（200字以内）：```# Phase 2 Tutorial Exit Artifact## 盲点1: <具体盲点，如 "GraphRAG 融合公式未定义">## 盲点2: <具体盲点，如 "MultiDiGraph 方向性语义不清">## 盲点3: <具体盲点，如 "向量语义相似 vs 推荐相关性混淆">## 推荐复习单元: <列出 1-2 个相关单元路径>## 下一步行动: <具体 drill 或 reading 条目>```**提交检查清单**：- [ ] student_model_p2.json 中 mastery 各字段已更新- [ ] blindspots 列表 >= 2 条- [ ] exit_artifact.md 已生成（200字以内）- [ ] Feed-Forward 的 re-drill 任务已加入本周计划> **Tutorial 闭环**：Oxford tutorial 的核心不是'老师讲了多少'，而是'学生被逼想了多少'。本仿真用静态 Socratic 分支模拟这一过程，不调真实 LLM API，确保 reproducible 与 anti-stall。